# 陈小群战法 - 三板加速术与持仓管理

> **策略来源**: 陈小群游资战法  
> **可靠性评级**: B级（中高可靠性）  
> **更新时间**: 2026-01-14

---

## 📊 功能说明

本Notebook用于执行三板加速术、持仓监控和止盈止损判断，是陈小群战法的持仓管理阶段。

### 三板加速术（40%加仓仓）

**确认条件**：
1. ✅ 在首板、二板基础上，第三天继续涨停
2. ✅ 第三板出现缩量涨停或量能持续放大
3. ✅ 板块效应持续增强（板块内涨停数量不减少）
4. ✅ 分时走势稳健（不频繁开板）

### 持仓监控指标

| 指标 | 健康标准 | 风险信号 |
|-----|---------|--------|
| **封单量** | 封单/流通市值 > 2% | < 1% 需警惕 |
| **板块效应** | 板块内涨停数 ≥ 3只 | < 2只需减仓 |
| **连板高度** | 市场最高连板 ≥ 当前板数 | 低于当前板数需警惕 |
| **炸板率** | < 25% | > 30% 需减仓 |

### 止盈止损策略

| 情况 | 操作 | 止损/止盈点 |
|-----|-----|------------|
| **首板失败** | 次日不涨停 | 止损 -5% |
| **二板失败** | 跌破开盘价 | 止损 -7% |
| **三板失败** | 跌破开盘价 | 止损 -10% |
| **见顶信号** | 板块效应减弱 | 及时止盈 |
| **龙头见顶** | 炸板率>30% | 大幅减仓 |

---

## 🔧 环境初始化

In [12]:
# 设置输出默认可滚动（限制最大高度）
from IPython.display import HTML, display

display(HTML("""
<style>
    .jp-OutputArea-output {
        max-height: 600px !important;
        overflow-y: auto !important;
    }
    .jp-Cell-outputArea {
        max-height: 600px !important;
        overflow-y: auto !important;
    }
    .jp-OutputArea-child {
        max-height: 600px !important;
        overflow-y: auto !important;
    }
</style>
"""))
print("✅ 输出区域已设置为可滚动（最大高度600px）")

✅ 输出区域已设置为可滚动（最大高度600px）


In [13]:
"""
环境初始化：设置项目路径和导入必要的库
"""

import sys
from pathlib import Path

# 自动检测项目根目录
current_dir = Path.cwd()
project_root = None
for parent in [current_dir] + list(current_dir.parents):
    if (parent / 'core').exists() and (parent / 'config').exists():
        project_root = parent
        break

if project_root is None:
    project_root = Path('/home/taotao/.cursor/worktrees/TRQuant/ope')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✅ 项目根目录: {project_root}")

# 导入必要的库
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✅ 基础库导入完成")

✅ 项目根目录: /home/taotao/.cursor/worktrees/TRQuant/ope
✅ 基础库导入完成


In [14]:
"""
初始化数据源：JQData和AKShare
"""

# 初始化JQData
jq = None
jq_status = "❌ 未连接"
try:
    import jqdatasdk as jq
    from config.config_manager import get_config_manager
    
    cm = get_config_manager()
    jq_config = cm.get_config('jqdata')
    jq.auth(jq_config['username'], jq_config['password'])
    
    if jq.is_auth():
        jq_status = "✅ 已连接"
        remaining = jq.get_query_count()
        print(f"JQData状态: {jq_status}")
        print(f"   剩余查询次数: {remaining.get('spare', 'N/A')}")
except Exception as e:
    print(f"JQData状态: {jq_status}")
    print(f"   错误: {str(e)[:100]}")

# 初始化AKShare
ak = None
ak_status = "❌ 未导入"
try:
    import akshare as ak
    ak_status = "✅ 已导入"
    print(f"AKShare状态: {ak_status}")
except Exception as e:
    print(f"AKShare状态: {ak_status}")
    print(f"   错误: {str(e)[:100]}")

print("\n" + "=" * 50)

JQData状态: ✅ 已连接
   剩余查询次数: 171886988
AKShare状态: ✅ 已导入



## 📥 读取第二步的选股结果

In [15]:
"""
读取第二步的选股结果（首板候选、二板潜力、龙头股票）
"""

print("=" * 80)
print("📥 读取第二步的选股结果")
print("=" * 80)

# 初始化变量
first_board_candidates = None
second_board_candidates = None
dragon_stocks = None
emotion_cycle = None
position = None
strategy = None
limit_up_count = None
max_height = None
zhaban_rate = None

# 从MongoDB读取最新结果
try:
    from core.notebook_result_manager import NotebookResultManager
    
    manager = NotebookResultManager(
        strategy_name="chen_xiaoqun_strategy",
        notebook_name="02_stock_selection"
    )
    
    # 获取最新结果
    latest_result = manager.get_latest_result()
    if latest_result:
        run_id = latest_result.get('run_id')
        if run_id:
            result = manager.load_result(run_id)
            if result:
                # 读取情绪周期信息
                emotion_cycle = result.get('emotion_cycle')
                position = result.get('position')
                strategy = result.get('strategy')
                limit_up_count = result.get('limit_up_count')
                max_height = result.get('max_height')
                zhaban_rate = result.get('zhaban_rate')
                
                # 读取选股结果
                first_board_data = result.get('first_board_candidates', [])
                if first_board_data:
                    first_board_candidates = pd.DataFrame(first_board_data)
                
                second_board_data = result.get('second_board_candidates', [])
                if second_board_data:
                    second_board_candidates = pd.DataFrame(second_board_data)
                
                dragon_data = result.get('dragon_stocks', [])
                if dragon_data:
                    dragon_stocks = pd.DataFrame(dragon_data)
                
                print(f"✅ 从MongoDB读取到第二步的最新结果（运行ID: {run_id}）")
                print(f"   运行日期: {latest_result.get('run_date', '')} {latest_result.get('run_time', '')}")
                print(f"\n📊 市场环境信息:")
                print(f"   情绪周期: {emotion_cycle}")
                print(f"   建议仓位: {position}")
                print(f"   推荐策略: {strategy}")
                if limit_up_count is not None:
                    print(f"   涨停家数: {limit_up_count}只")
                if max_height is not None:
                    print(f"   最高连板: {max_height}板")
                if zhaban_rate is not None:
                    print(f"   炸板率: {zhaban_rate:.2f}%")
    else:
        print("⚠️  未找到第二步的结果，请先运行 02_stock_selection.ipynb")
except Exception as e:
    print(f"⚠️  从MongoDB读取失败: {str(e)[:100]}")

# 显示选股结果摘要
print("\n" + "=" * 80)
print("📊 选股结果摘要")
print("=" * 80)

if first_board_candidates is not None and not first_board_candidates.empty:
    print(f"\n✅ 首板候选股票: {len(first_board_candidates)} 只")
    display_cols = ['代码', '名称', '涨跌幅', '换手率', '封板资金占比(%)']
    available_cols = [c for c in display_cols if c in first_board_candidates.columns]
    if available_cols:
        print(first_board_candidates[available_cols].head(5).to_string(index=False))
else:
    print("\n⚠️  未找到首板候选股票")
    first_board_candidates = pd.DataFrame()

if second_board_candidates is not None and not second_board_candidates.empty:
    print(f"\n✅ 二板潜力股票: {len(second_board_candidates)} 只")
    display_cols = ['代码', '名称', '二板潜力', '评分']
    available_cols = [c for c in display_cols if c in second_board_candidates.columns]
    if available_cols:
        print(second_board_candidates[available_cols].head(5).to_string(index=False))
else:
    print("\n⚠️  未找到二板潜力股票")
    second_board_candidates = pd.DataFrame()

if dragon_stocks is not None and not dragon_stocks.empty:
    print(f"\n✅ 龙头股票: {len(dragon_stocks)} 只")
    display_cols = ['代码', '名称', '连板数', '涨跌幅']
    available_cols = [c for c in display_cols if c in dragon_stocks.columns]
    if available_cols:
        print(dragon_stocks[available_cols].head(5).to_string(index=False))
else:
    print("\n⚠️  未找到龙头股票")
    dragon_stocks = pd.DataFrame()

📥 读取第二步的选股结果
✅ 从MongoDB读取到第二步的最新结果（运行ID: 20260114_212145）
   运行日期: 2026-01-14 21:21:45

📊 市场环境信息:
   情绪周期: 过热期
   建议仓位: 30-50%
   推荐策略: 逐步减仓
   涨停家数: 102只
   最高连板: 5板
   炸板率: 4.67%

📊 选股结果摘要

⚠️  未找到首板候选股票

⚠️  未找到二板潜力股票

⚠️  未找到龙头股票


## 🔍 获取最新市场数据

In [16]:
"""
获取最新的涨停板数据，用于三板加速术分析
"""

print("=" * 80)
print("🔍 获取最新市场数据")
print("=" * 80)

today = datetime.now().strftime('%Y%m%d')
print(f"\n📅 当前日期: {today}")

# 获取涨停板数据
limit_up_data = None
max_height_today = None
limit_up_count_today = 0
try:
    limit_up_data = ak.stock_zt_pool_em(date=today)
    if limit_up_data is not None and not limit_up_data.empty:
        limit_up_count_today = len(limit_up_data)
        print(f"✅ 获取到 {limit_up_count_today} 只涨停股票")
        
        # 计算最高连板
        if '连板数' in limit_up_data.columns:
            max_height_today = limit_up_data['连板数'].max()
            print(f"   今日最高连板: {max_height_today}板")
    else:
        print("⚠️  今日无涨停数据（可能是非交易日）")
except Exception as e:
    print(f"❌ 获取涨停数据失败: {str(e)[:100]}")

# 获取炸板数据
zhaban_data = None
zhaban_rate_today = None
zhaban_count = 0
try:
    zhaban_data = ak.stock_zt_pool_zbgc_em(date=today)
    if zhaban_data is not None and not zhaban_data.empty:
        zhaban_count = len(zhaban_data)
        if limit_up_data is not None and not limit_up_data.empty:
            total_zt = len(limit_up_data) + zhaban_count
            if total_zt > 0:
                zhaban_rate_today = (zhaban_count / total_zt) * 100
                print(f"   今日炸板率: {zhaban_rate_today:.2f}% ({zhaban_count}只炸板)")
except Exception as e:
    print(f"⚠️  获取炸板数据失败: {str(e)[:50]}")

# 数据一致性检查（与第二步保存的数据对比）
print("\n" + "-" * 60)
print("📊 数据一致性检查（与第二步对比）")
print("-" * 60)

data_consistent = True
if limit_up_count is not None and limit_up_count_today > 0:
    diff = abs(limit_up_count_today - limit_up_count)
    if diff > 5:
        print(f"⚠️  涨停家数有差异: 第二步 {limit_up_count}只 vs 今日 {limit_up_count_today}只")
        print(f"   💡 可能原因：1) 不同时间运行 2) 数据源延迟更新")
        data_consistent = False
    else:
        print(f"✅ 涨停家数一致: {limit_up_count_today}只")

if zhaban_rate is not None and zhaban_rate_today is not None:
    diff = abs(zhaban_rate_today - zhaban_rate)
    if diff > 5:
        print(f"⚠️  炸板率有差异: 第二步 {zhaban_rate:.2f}% vs 今日 {zhaban_rate_today:.2f}%")
        print(f"   💡 说明：炸板率在盘中会动态变化，收盘后数据才稳定")
        data_consistent = False
    else:
        print(f"✅ 炸板率一致: {zhaban_rate_today:.2f}%")

if max_height is not None and max_height_today is not None:
    if max_height_today != max_height:
        print(f"⚠️  最高连板有差异: 第二步 {max_height}板 vs 今日 {max_height_today}板")
        data_consistent = False
    else:
        print(f"✅ 最高连板一致: {max_height_today}板")

if data_consistent:
    print("\n✅ 数据一致，分析结果可靠")
else:
    print("\n⚠️  数据有差异，建议重新运行第一、二步获取最新数据")

print("\n" + "=" * 50)

🔍 获取最新市场数据

📅 当前日期: 20260114


✅ 获取到 102 只涨停股票
   今日最高连板: 5板
   今日炸板率: 36.65% (59只炸板)

------------------------------------------------------------
📊 数据一致性检查（与第二步对比）
------------------------------------------------------------
✅ 涨停家数一致: 102只
⚠️  炸板率有差异: 第二步 4.67% vs 今日 36.65%
   💡 说明：炸板率在盘中会动态变化，收盘后数据才稳定
✅ 最高连板一致: 5板

⚠️  数据有差异，建议重新运行第一、二步获取最新数据



## 🚀 三板加速术分析

In [17]:
"""
三板加速术：分析二板股票的三板潜力

确认条件：
1. 在首板、二板基础上，第三天继续涨停
2. 第三板出现缩量涨停或量能持续放大
3. 板块效应持续增强
4. 分时走势稳健
"""

print("=" * 80)
print("🚀 三板加速术分析")
print("=" * 80)

# 检查情绪周期
if emotion_cycle is not None:
    print(f"\n📊 当前情绪周期: {emotion_cycle}")
    if emotion_cycle == "退潮期":
        print("\n⚠️  当前为退潮期，不建议执行三板加速术")
        print("   💡 建议：空仓等待，保持耐心")
    elif emotion_cycle == "过热期":
        print("\n⚠️  当前为过热期，三板加速术风险较高")
        print("   💡 建议：控制仓位，设置严格止损")
    else:
        print(f"\n✅ 当前周期适合执行三板加速术")

# 三板潜力分析
third_board_candidates = []
third_board_df = pd.DataFrame()

if limit_up_data is not None and not limit_up_data.empty:
    print("\n" + "-" * 60)
    print("📊 分析连板股票的三板潜力")
    print("-" * 60)
    
    # 筛选2板以上的股票（可能是今日冲击三板或更高）
    if '连板数' in limit_up_data.columns:
        two_plus_boards = limit_up_data[limit_up_data['连板数'] >= 2].copy()
        
        if not two_plus_boards.empty:
            print(f"\n找到 {len(two_plus_boards)} 只2板及以上股票")
            
            for idx, row in two_plus_boards.iterrows():
                code = row.get('代码', '')
                name = row.get('名称', '')
                board_count = row.get('连板数', 0)
                turnover = row.get('换手率', 0)
                sector = row.get('所属行业', '')
                limit_amount = row.get('封板资金', 0)
                
                # 评估三板潜力
                score = 0
                factors = []
                
                # 条件1: 连板数（2板冲3板、3板冲4板等）
                if board_count == 2:
                    score += 1
                    factors.append(f"当前2板，今日冲击3板 ✅")
                elif board_count >= 3:
                    score += 1.5
                    factors.append(f"当前{board_count}板，连板强势 ✅✅")
                
                # 条件2: 换手率分析（缩量或放量）
                if turnover < 10:
                    score += 1
                    factors.append(f"换手率{turnover:.2f}%，缩量涨停（筹码锁定）✅")
                elif turnover > 25:
                    score += 0.5
                    factors.append(f"换手率{turnover:.2f}%，放量涨停（新资金入场）⚠️")
                else:
                    factors.append(f"换手率{turnover:.2f}%（正常）")
                
                # 条件3: 板块效应（检查同板块涨停数）
                sector_limit_up_count = 0
                if sector and '所属行业' in limit_up_data.columns:
                    sector_limit_up_count = len(limit_up_data[limit_up_data['所属行业'] == sector])
                    if sector_limit_up_count >= 3:
                        score += 1
                        factors.append(f"板块效应强（{sector}板块{sector_limit_up_count}只涨停）✅")
                    elif sector_limit_up_count >= 2:
                        score += 0.5
                        factors.append(f"板块效应中等（{sector}板块{sector_limit_up_count}只涨停）⚠️")
                    else:
                        factors.append(f"板块效应弱（{sector}板块{sector_limit_up_count}只涨停）❌")
                
                # 条件4: 封板资金
                if limit_amount:
                    limit_amount_yi = limit_amount / 1e8
                    if limit_amount_yi >= 2:
                        score += 1
                        factors.append(f"封板资金{limit_amount_yi:.2f}亿（资金共识强）✅")
                    elif limit_amount_yi >= 1:
                        score += 0.5
                        factors.append(f"封板资金{limit_amount_yi:.2f}亿（资金共识中等）⚠️")
                    else:
                        factors.append(f"封板资金{limit_amount_yi:.2f}亿（资金共识弱）❌")
                
                # 确定三板潜力等级
                if score >= 3:
                    potential = "高"
                elif score >= 2:
                    potential = "中"
                else:
                    potential = "低"
                
                third_board_candidates.append({
                    '代码': code,
                    '名称': name,
                    '当前连板': board_count,
                    '换手率': turnover,
                    '所属行业': sector,
                    '板块涨停数': sector_limit_up_count,
                    '评分': score,
                    '三板潜力': potential,
                    '评估因素': factors
                })
            
            # 转换为DataFrame并排序
            third_board_df = pd.DataFrame(third_board_candidates)
            third_board_df = third_board_df.sort_values('评分', ascending=False)
            
            # 显示结果
            print("\n" + "=" * 80)
            print("📊 三板潜力分析结果")
            print("=" * 80)
            
            # 高潜力
            high_potential = third_board_df[third_board_df['三板潜力'] == '高']
            if not high_potential.empty:
                print(f"\n🔥 高潜力股票（{len(high_potential)}只）：")
                for _, stock in high_potential.iterrows():
                    print(f"\n   {stock['名称']} ({stock['代码']})")
                    print(f"   当前: {stock['当前连板']}板 | 换手率: {stock['换手率']:.2f}%")
                    print(f"   板块: {stock['所属行业']}（{stock['板块涨停数']}只涨停）")
                    print(f"   评分: {stock['评分']:.1f}分 | 三板潜力: {stock['三板潜力']}")
                    print(f"   分析: ")
                    for factor in stock['评估因素']:
                        print(f"      - {factor}")
            
            # 中等潜力
            medium_potential = third_board_df[third_board_df['三板潜力'] == '中']
            if not medium_potential.empty:
                print(f"\n⚠️  中等潜力股票（{len(medium_potential)}只）：")
                for _, stock in medium_potential.head(3).iterrows():
                    print(f"   {stock['名称']} ({stock['代码']}) - {stock['当前连板']}板 - 评分{stock['评分']:.1f}")
            
            # 低潜力
            low_potential = third_board_df[third_board_df['三板潜力'] == '低']
            if not low_potential.empty:
                print(f"\n❌ 低潜力股票（{len(low_potential)}只）：不建议参与")
            
        else:
            print("\n⚠️  今日无2板及以上股票，暂无三板机会")
    else:
        print("\n⚠️  数据中无连板数信息")
else:
    print("\n⚠️  无涨停数据，无法进行三板分析")

🚀 三板加速术分析

📊 当前情绪周期: 过热期

⚠️  当前为过热期，三板加速术风险较高
   💡 建议：控制仓位，设置严格止损

------------------------------------------------------------
📊 分析连板股票的三板潜力
------------------------------------------------------------

找到 19 只2板及以上股票

📊 三板潜力分析结果

🔥 高潜力股票（11只）：

   美年健康 (002044)
   当前: 4板 | 换手率: 8.26%
   板块: 医疗服务（3只涨停）
   评分: 4.5分 | 三板潜力: 高
   分析: 
      - 当前4板，连板强势 ✅✅
      - 换手率8.26%，缩量涨停（筹码锁定）✅
      - 板块效应强（医疗服务板块3只涨停）✅
      - 封板资金6.81亿（资金共识强）✅

   三维通信 (002115)
   当前: 4板 | 换手率: 1.01%
   板块: 互联网服（17只涨停）
   评分: 4.5分 | 三板潜力: 高
   分析: 
      - 当前4板，连板强势 ✅✅
      - 换手率1.01%，缩量涨停（筹码锁定）✅
      - 板块效应强（互联网服板块17只涨停）✅
      - 封板资金15.71亿（资金共识强）✅

   三江购物 (601116)
   当前: 3板 | 换手率: 1.13%
   板块: 商业百货（3只涨停）
   评分: 4.5分 | 三板潜力: 高
   分析: 
      - 当前3板，连板强势 ✅✅
      - 换手率1.13%，缩量涨停（筹码锁定）✅
      - 板块效应强（商业百货板块3只涨停）✅
      - 封板资金2.12亿（资金共识强）✅

   人民网 (603000)
   当前: 3板 | 换手率: 9.92%
   板块: 文化传媒（7只涨停）
   评分: 4.5分 | 三板潜力: 高
   分析: 
      - 当前3板，连板强势 ✅✅
      - 换手率9.92%，缩量涨停（筹码锁定）✅
      - 板块效应强（文化传媒板块7只涨停）✅
      - 封

## 📈 持仓监控

In [18]:
"""
持仓监控：监控持仓股票的关键指标

如果第二步没有选出候选股票，则使用三板潜力分析中的高潜力股票作为监控对象
"""

print("=" * 80)
print("📈 持仓监控")
print("=" * 80)

def monitor_position(stock_code, stock_name, limit_up_data):
    """监控单只股票的持仓状态"""
    status = 'holding'
    risk_level = 0
    signals = []
    
    if limit_up_data is None or limit_up_data.empty:
        return 'warning', 50, ['无法获取市场数据']
    
    stock_in_zt = limit_up_data[limit_up_data['代码'] == stock_code]
    
    if stock_in_zt.empty:
        status = 'warning'
        risk_level += 30
        signals.append("⚠️  股票今日未涨停，需要关注")
    else:
        stock_info = stock_in_zt.iloc[0]
        signals.append(f"✅ 股票今日涨停（连板数: {stock_info.get('连板数', 'N/A')}）")
        
        limit_amount = stock_info.get('封板资金', 0)
        if limit_amount:
            limit_amount_yi = limit_amount / 1e8
            if limit_amount_yi < 0.5:
                risk_level += 20
                signals.append(f"⚠️  封板资金较少（{limit_amount_yi:.2f}亿）")
            else:
                signals.append(f"✅ 封板资金充足（{limit_amount_yi:.2f}亿）")
        
        sector = stock_info.get('所属行业', '')
        if sector and '所属行业' in limit_up_data.columns:
            sector_count = len(limit_up_data[limit_up_data['所属行业'] == sector])
            if sector_count < 2:
                risk_level += 15
                signals.append(f"⚠️  板块效应减弱（{sector}仅{sector_count}只涨停）")
            else:
                signals.append(f"✅ 板块效应良好（{sector}有{sector_count}只涨停）")
    
    if risk_level >= 50:
        status = 'exit'
    elif risk_level >= 30:
        status = 'warning'
    
    return status, risk_level, signals

# 监控股票
position_status_list = []

# 首先尝试监控第二步的候选股票
has_second_step_data = False

if second_board_candidates is not None and not second_board_candidates.empty:
    has_second_step_data = True
    print("\n📊 监控二板候选股票：")
    if '二板潜力' in second_board_candidates.columns:
        high_potential_stocks = second_board_candidates[second_board_candidates['二板潜力'] == '高']
    else:
        high_potential_stocks = second_board_candidates.head(5)
    
    for _, stock in high_potential_stocks.iterrows():
        code = stock.get('代码', '')
        name = stock.get('名称', '')
        if code and name:
            status, risk, signals = monitor_position(code, name, limit_up_data)
            status_icon = '🟢' if status == 'holding' else '🟡' if status == 'warning' else '🔴'
            print(f"\n{status_icon} {name} ({code}) - 风险: {risk}/100")
            for signal in signals:
                print(f"   {signal}")
            position_status_list.append({'代码': code, '名称': name, '状态': status, '风险等级': risk, '信号': signals})

if dragon_stocks is not None and not dragon_stocks.empty:
    has_second_step_data = True
    print("\n🐉 监控龙头股票：")
    for _, stock in dragon_stocks.head(3).iterrows():
        code = stock.get('代码', '')
        name = stock.get('名称', '')
        if code and name:
            status, risk, signals = monitor_position(code, name, limit_up_data)
            status_icon = '🟢' if status == 'holding' else '🟡' if status == 'warning' else '🔴'
            print(f"\n{status_icon} {name} ({code}) - 风险: {risk}/100")
            for signal in signals:
                print(f"   {signal}")
            position_status_list.append({'代码': code, '名称': name, '状态': status, '风险等级': risk, '信号': signals, '类型': '龙头'})

# 如果第二步没有数据，使用三板潜力分析中的高潜力股票
if not has_second_step_data and 'third_board_df' in dir() and not third_board_df.empty:
    print("\n⚠️  第二步未选出候选股票，使用三板潜力分析结果进行监控")
    high_potential = third_board_df[third_board_df['三板潜力'] == '高']
    
    if not high_potential.empty:
        print(f"\n🔥 监控三板高潜力股票（{len(high_potential)}只）：")
        for _, stock in high_potential.head(5).iterrows():
            code = stock.get('代码', '')
            name = stock.get('名称', '')
            if code and name:
                status, risk, signals = monitor_position(code, name, limit_up_data)
                status_icon = '🟢' if status == 'holding' else '🟡' if status == 'warning' else '🔴'
                print(f"\n{status_icon} {name} ({code}) - {stock.get('当前连板', 0)}板 - 风险: {risk}/100")
                for signal in signals:
                    print(f"   {signal}")
                position_status_list.append({
                    '代码': code, 
                    '名称': name, 
                    '状态': status, 
                    '风险等级': risk, 
                    '信号': signals,
                    '当前连板': stock.get('当前连板', 0),
                    '三板潜力': stock.get('三板潜力', ''),
                    '类型': '三板候选'
                })
    else:
        print("\n⚠️  暂无高潜力股票需要监控")
elif not has_second_step_data:
    print("\n⚠️  暂无候选股票需要监控")

position_status_df = pd.DataFrame(position_status_list) if position_status_list else pd.DataFrame()

# 显示监控摘要
if not position_status_df.empty:
    print("\n" + "-" * 60)
    print("📋 监控摘要")
    print("-" * 60)
    holding_count = len(position_status_df[position_status_df['状态'] == 'holding'])
    warning_count = len(position_status_df[position_status_df['状态'] == 'warning'])
    exit_count = len(position_status_df[position_status_df['状态'] == 'exit'])
    print(f"   🟢 持有: {holding_count}只")
    print(f"   🟡 警告: {warning_count}只")
    print(f"   🔴 退出: {exit_count}只")

📈 持仓监控

⚠️  第二步未选出候选股票，使用三板潜力分析结果进行监控

🔥 监控三板高潜力股票（11只）：

🟢 美年健康 (002044) - 4板 - 风险: 0/100
   ✅ 股票今日涨停（连板数: 4）
   ✅ 封板资金充足（6.81亿）
   ✅ 板块效应良好（医疗服务有3只涨停）

🟢 三维通信 (002115) - 4板 - 风险: 0/100
   ✅ 股票今日涨停（连板数: 4）
   ✅ 封板资金充足（15.71亿）
   ✅ 板块效应良好（互联网服有17只涨停）

🟢 三江购物 (601116) - 3板 - 风险: 0/100
   ✅ 股票今日涨停（连板数: 3）
   ✅ 封板资金充足（2.12亿）
   ✅ 板块效应良好（商业百货有3只涨停）

🟢 人民网 (603000) - 3板 - 风险: 0/100
   ✅ 股票今日涨停（连板数: 3）
   ✅ 封板资金充足（4.14亿）
   ✅ 板块效应良好（文化传媒有7只涨停）

🟢 三变科技 (002112) - 2板 - 风险: 0/100
   ✅ 股票今日涨停（连板数: 2）
   ✅ 封板资金充足（2.64亿）
   ✅ 板块效应良好（电网设备有5只涨停）

------------------------------------------------------------
📋 监控摘要
------------------------------------------------------------
   🟢 持有: 5只
   🟡 警告: 0只
   🔴 退出: 0只


## ⚠️ 止盈止损判断

In [19]:
"""
止盈止损判断：根据市场情况给出操作建议
"""

print("=" * 80)
print("⚠️  止盈止损判断")
print("=" * 80)

# 市场风险评估
market_risk = 0
market_signals = []

if zhaban_rate_today is not None:
    if zhaban_rate_today > 30:
        market_risk += 30
        market_signals.append(f"🔴 炸板率过高 ({zhaban_rate_today:.2f}%)，市场情绪不稳")
    elif zhaban_rate_today > 25:
        market_risk += 15
        market_signals.append(f"🟡 炸板率偏高 ({zhaban_rate_today:.2f}%)")
    else:
        market_signals.append(f"🟢 炸板率正常 ({zhaban_rate_today:.2f}%)")

if limit_up_count is not None and limit_up_data is not None:
    today_count = len(limit_up_data)
    if today_count < limit_up_count * 0.7:
        market_risk += 25
        market_signals.append(f"🔴 涨停家数大幅下降 ({limit_up_count} → {today_count})")
    else:
        market_signals.append(f"🟢 涨停家数稳定 (今日{today_count}只)")

if max_height is not None and max_height_today is not None:
    if max_height_today < max_height - 1:
        market_risk += 20
        market_signals.append(f"🔴 连板高度下降 ({max_height}板 → {max_height_today}板)")
    else:
        market_signals.append(f"🟢 连板高度稳定 (今日最高{max_height_today}板)")

if emotion_cycle == "过热期":
    market_risk += 20
    market_signals.append("🟡 当前处于过热期，注意及时止盈")
elif emotion_cycle == "退潮期":
    market_risk += 40
    market_signals.append("🔴 当前处于退潮期，建议清仓观望")

print(f"\n市场整体风险等级: {market_risk}/100")
risk_level_text = "🔴 高风险" if market_risk >= 50 else "🟡 中等风险" if market_risk >= 30 else "🟢 低风险"
print(f"风险评级: {risk_level_text}")
for signal in market_signals:
    print(f"   {signal}")

# 操作建议
print("\n" + "=" * 80)
print("📋 操作建议")
print("=" * 80)

operation_advice = []
if market_risk >= 50:
    print("\n🔴 市场整体建议：减仓/止盈，建议仓位30%以下")
    operation_advice.append({'类型': '市场', '操作': '减仓/止盈', '建议仓位': '30%以下'})
elif market_risk >= 30:
    print("\n🟡 市场整体建议：持仓观察，建议仓位50%左右")
    operation_advice.append({'类型': '市场', '操作': '持仓观察', '建议仓位': '50%左右'})
else:
    print("\n🟢 市场整体建议：正常持仓，根据策略执行")
    operation_advice.append({'类型': '市场', '操作': '正常持仓', '建议仓位': '根据策略执行'})

operation_advice_df = pd.DataFrame(operation_advice)

⚠️  止盈止损判断

市场整体风险等级: 50/100
风险评级: 🔴 高风险
   🔴 炸板率过高 (36.65%)，市场情绪不稳
   🟢 涨停家数稳定 (今日102只)
   🟢 连板高度稳定 (今日最高5板)
   🟡 当前处于过热期，注意及时止盈

📋 操作建议

🔴 市场整体建议：减仓/止盈，建议仓位30%以下


## 📊 可视化分析

In [20]:
"""
可视化持仓管理结果
"""
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("=" * 80)
print("📊 可视化分析")
print("=" * 80)

has_third_board = not third_board_df.empty if 'third_board_df' in dir() else False
has_position_status = not position_status_df.empty if 'position_status_df' in dir() else False
has_market_data = limit_up_data is not None and not limit_up_data.empty

if not (has_third_board or has_position_status or has_market_data):
    print("\n⚠️  暂无足够数据生成可视化图表")
else:
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('三板潜力分布', '市场风险仪表盘', '持仓状态分布', '板块效应分析'),
        specs=[[{"type": "bar"}, {"type": "indicator"}], [{"type": "bar"}, {"type": "pie"}]],
        vertical_spacing=0.18, horizontal_spacing=0.12
    )
    
    # 1. 三板潜力分布
    if has_third_board:
        potential_counts = third_board_df['三板潜力'].value_counts()
        colors = ['green' if p == '高' else 'gold' if p == '中' else 'red' for p in potential_counts.index]
        fig.add_trace(go.Bar(x=potential_counts.index, y=potential_counts.values, marker_color=colors, showlegend=False), row=1, col=1)
        fig.update_xaxes(title_text="潜力等级", row=1, col=1)
        fig.update_yaxes(title_text="股票数量", row=1, col=1)
    
    # 2. 市场风险仪表盘
    fig.add_trace(go.Indicator(
        mode="gauge+number", value=market_risk,
        gauge={'axis': {'range': [0, 100]}, 'bar': {'color': "darkblue"},
               'steps': [{'range': [0, 30], 'color': "lightgreen"}, {'range': [30, 50], 'color': "yellow"}, {'range': [50, 100], 'color': "lightcoral"}]}
    ), row=1, col=2)
    
    # 3. 持仓状态分布
    if has_position_status:
        status_counts = position_status_df['状态'].value_counts()
        status_colors = {'holding': 'green', 'warning': 'gold', 'exit': 'red'}
        fig.add_trace(go.Bar(x=[{'holding': '持有', 'warning': '警告', 'exit': '退出'}.get(s, s) for s in status_counts.index],
                            y=status_counts.values, marker_color=[status_colors.get(s, 'gray') for s in status_counts.index], showlegend=False), row=2, col=1)
    
    # 4. 板块效应分析
    if has_market_data and '所属行业' in limit_up_data.columns:
        sector_counts = limit_up_data['所属行业'].value_counts().head(8)
        fig.add_trace(go.Pie(labels=sector_counts.index, values=sector_counts.values, hole=0.4, textinfo='label+value'), row=2, col=2)
    
    fig.update_layout(
        title_text=f"陈小群战法 - 持仓管理分析<br><span style='font-size:0.7em'>情绪周期: {emotion_cycle or '未知'} | 市场风险: {market_risk}/100</span>",
        height=700, showlegend=False
    )
    fig.show()
    print("\n✅ 可视化图表已生成")

📊 可视化分析



✅ 可视化图表已生成


## 💾 保存分析结果

In [21]:
"""
保存持仓管理分析结果到文件系统和MongoDB
"""
import json
import os

print("=" * 80)
print("💾 保存分析结果")
print("=" * 80)

save_result = {
    'analysis_date': datetime.now().strftime('%Y-%m-%d'),
    'analysis_time': datetime.now().strftime('%H:%M:%S'),
    'notebook_name': '03_position_management',
    'strategy_name': 'chen_xiaoqun_strategy',
    'emotion_cycle': emotion_cycle,
    'position': position,
    'strategy': strategy,
    'market_risk': market_risk,
    'market_signals': market_signals
}

# 添加三板潜力分析结果
if 'third_board_df' in dir() and not third_board_df.empty:
    third_board_records = []
    for _, row in third_board_df.iterrows():
        record = row.to_dict()
        if '评估因素' in record and isinstance(record['评估因素'], list):
            record['评估因素'] = [str(f) for f in record['评估因素']]
        third_board_records.append(record)
    save_result['third_board_candidates'] = third_board_records
    save_result['third_board_count'] = len(third_board_df)
    print(f"✅ 三板潜力分析: {len(third_board_df)} 只股票")

# 添加持仓监控结果
if 'position_status_df' in dir() and not position_status_df.empty:
    position_records = []
    for _, row in position_status_df.iterrows():
        record = row.to_dict()
        if '信号' in record and isinstance(record['信号'], list):
            record['信号'] = [str(s) for s in record['信号']]
        position_records.append(record)
    save_result['position_status'] = position_records
    save_result['position_count'] = len(position_status_df)
    print(f"✅ 持仓监控: {len(position_status_df)} 只股票")

# 添加操作建议
if 'operation_advice_df' in dir() and not operation_advice_df.empty:
    save_result['operation_advice'] = operation_advice_df.to_dict('records')
    print(f"✅ 操作建议: {len(operation_advice_df)} 条")

# 保存到MongoDB
try:
    from core.notebook_result_manager import NotebookResultManager
    manager = NotebookResultManager(strategy_name="chen_xiaoqun_strategy", notebook_name="03_position_management")
    save_info = manager.save_result(save_result)
    # 提取run_id（可能是dict或字符串）
    if isinstance(save_info, dict):
        run_id = save_info.get('run_id', str(save_info))
    else:
        run_id = str(save_info)
    print(f"\n✅ 结果已保存到MongoDB (运行ID: {run_id})")
except Exception as e:
    print(f"\n⚠️  保存到MongoDB失败: {str(e)[:100]}")

# 保存到文件系统
try:
    results_dir = project_root / 'notebooks' / 'research' / 'results' / 'chen_xiaoqun_strategy' / '03_position_management'
    results_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    result_file = results_dir / f'{timestamp}' / 'result.json'
    result_file.parent.mkdir(parents=True, exist_ok=True)
    with open(result_file, 'w', encoding='utf-8') as f:
        json.dump(save_result, f, ensure_ascii=False, indent=2, default=str)
    print(f"✅ 结果已保存到文件: {result_file}")
except Exception as e:
    print(f"\n⚠️  保存到文件失败: {str(e)[:100]}")

print("\n" + "=" * 80)
print("✅ 第三步（三板加速术与持仓管理）完成")
print("=" * 80)

💾 保存分析结果
✅ 三板潜力分析: 19 只股票
✅ 持仓监控: 5 只股票
✅ 操作建议: 1 条

✅ 结果已保存到MongoDB (运行ID: 20260114_220333)
✅ 结果已保存到文件: /home/taotao/.cursor/worktrees/TRQuant/ope/notebooks/research/results/chen_xiaoqun_strategy/03_position_management/20260114_090333/result.json

✅ 第三步（三板加速术与持仓管理）完成


## 📋 总结

### 本Notebook完成的工作：

1. **读取第二步选股结果**：从MongoDB读取首板候选、二板潜力、龙头股票

2. **三板加速术分析**：
   - 筛选2板及以上股票
   - 评估三板潜力（缩量/放量、板块效应、封板资金）
   - 分类为高/中/低潜力

3. **持仓监控**：
   - 监控股票是否仍在涨停
   - 检查封单量变化
   - 检查板块效应变化
   - 给出持仓状态（持有/警告/退出）

4. **止盈止损判断**：
   - 市场整体风险评估
   - 个股风险评估
   - 生成操作建议

5. **可视化分析**：
   - 三板潜力分布
   - 市场风险仪表盘
   - 持仓风险分布
   - 板块效应分析

6. **保存分析结果**：
   - 保存到MongoDB
   - 保存到文件系统

### 下一步：

- **04_backtest_validation.ipynb**：策略回测验证
  - 历史数据回测
  - 成功率统计
  - 收益率分析
  - 风险指标评估